In [1]:
# ----------------------------------------------------------------------
# Configuração inicial: cria o transacoes.csv usado para testar o notebook.
# 16 registros válidos (4 meses), 6 inválidos (um por critério) e
# 2 transações acima de LIMITE_SUSPEITO.
# ----------------------------------------------------------------------

import csv
dados = [
    {"id": "1", "data": "2026-01-05", "cliente_id": "CLI001", "tipo": "credito", "valor": "3512.47", "descricao": "Salário", "categoria": "salario"},
    {"id": "ab", "data": "2026-01-09", "cliente_id": "CLI002", "tipo": "debito", "valor": "187.35", "descricao": "Supermercado", "categoria": "compra"},
    {"id": "3", "data": "2026-01-14", "cliente_id": "CLI003", "tipo": "debito", "valor": "42.80", "descricao": "Uber", "categoria": "transporte"},
    {"id": "4", "data": "2026-01-22", "cliente_id": "CLI001", "tipo": "debito", "valor": "68.90", "descricao": "Restaurante", "categoria": "alimentacao"},
    {"id": "5", "data": "2026-02-02", "cliente_id": "CLI002", "tipo": "credito", "valor": "2789.60", "descricao": "Transferência recebida", "categoria": "transferencia"},
    {"id": "6", "data": "2026-02-09", "cliente_id": "CLI003", "tipo": "debito", "valor": "320.75", "descricao": "Farmácia", "categoria": "compra"},
    {"id": "7", "data": "2026-02-14", "cliente_id": "CLI001", "tipo": "credito", "valor": "15230.90", "descricao": "Transferência recebida", "categoria": "transferencia"},
    {"id": "8", "data": "2026-02-19", "cliente_id": "CLI002", "tipo": "debito", "valor": "289.99", "descricao": "Compra de criptomoeda", "categoria": "criptomoeda"},
    {"id": "9", "data": "2026-02-27", "cliente_id": "CLI003", "tipo": "debito", "valor": "54.60", "descricao": "Ônibus", "categoria": "transporte"},
    {"id": "10", "data": "2026-03-03", "cliente_id": "CLI001", "tipo": "credito", "valor": "3512.47", "descricao": "Salário", "categoria": "salario"},
    {"id": "11", "data": "2026-03-06", "cliente_id": "CLI002", "tipo": "debito", "valor": "218.60", "descricao": "Supermercado", "categoria": "compra"},
    {"id": "12", "data": "2026-03-11", "cliente_id": "CLI003", "tipo": "debito", "valor": "12045.80", "descricao": "Compra de criptomoeda", "categoria": "criptomoeda"},
    {"id": "13", "data": "2026-03-17", "cliente_id": "CLI001", "tipo": "debito", "valor": "76.42", "descricao": "Lanchonete", "categoria": "alimentacao"},
    {"id": "14", "data": "2026-03-21", "cliente_id": "CLI002", "tipo": "credito", "valor": "2789.60", "descricao": "Salário", "categoria": "salario"},
    {"id": "15", "data": "2026-03-29", "cliente_id": "CLI003", "tipo": "debito", "valor": "33.90", "descricao": "Táxi", "categoria": "transporte"},
    {"id": "16", "data": "2026-04-04", "cliente_id": "CLI001", "tipo": "debito", "valor": "94.30", "descricao": "Restaurante", "categoria": "alimentacao"},
    {"id": "17", "data": "2026-02-03", "cliente_id": "", "tipo": "debito", "valor": "204.35", "descricao": "Sem cliente", "categoria": "transferencia"},
    {"id": "", "data": "2026-02-08", "cliente_id": "CLI002", "tipo": "credito", "valor": "315.20", "descricao": "Sem id", "categoria": "salario"},
    {"id": "19", "data": "2026/15/03", "cliente_id": "CLI003", "tipo": "debito", "valor": "80.60", "descricao": "Data errada", "categoria": "compra"},
    {"id": "20", "data": "2026-01-20", "cliente_id": "CLI001", "tipo": "debito", "valor": "abc", "descricao": "Erro no valor", "categoria": "compra"},
    {"id": "21", "data": "2026-03-20", "cliente_id": "CLI001", "tipo": "pix", "valor": "150.90", "descricao": "Tipo inválido", "categoria": "transferencia"},
    {"id": "22", "data": "2026-03-25", "cliente_id": "CLI002", "tipo": "debito", "valor": "-75.40", "descricao": "Valor negativo", "categoria": "alimentacao"},
]

with open("transacoes.csv", mode="w", newline="", encoding="utf-8") as arquivo:
    campos = ["id", "data", "cliente_id", "tipo", "valor", "descricao", "categoria"]
    escritor = csv.DictWriter(arquivo, fieldnames=campos)
    escritor.writeheader()
    escritor.writerows(dados)

print("Arquivo transacoes.csv criado com sucesso!")

Arquivo transacoes.csv criado com sucesso!


In [2]:
# ----------------------------------------------------------------------
# Imports e constantes usadas em todo o notebook.
# ----------------------------------------------------------------------
import json
import csv
from datetime import datetime
LIMITE_SUSPEITO = 10000.00

In [3]:
# ----------------------------------------------------------------------
# Requisito: "Leitura do arquivo CSV com módulo nativo"
# 1. Utilize o módulo csv nativo do Python (sem pandas) para leitura.
# 2. Use csv.DictReader para acessar as colunas pelo nome.
# 3. Trate o caso em que o arquivo não existe (FileNotFoundError).
# ----------------------------------------------------------------------
def ler_csv(nome_arquivo_ler):

    dados_ler = []
    try:
        with open(nome_arquivo_ler, mode="r", encoding="utf-8") as arquivo:
            leitor = csv.DictReader(arquivo)
            for linha in leitor:
                dados_ler.append(linha)
    except FileNotFoundError:
        print(f"Error no arquivo '{nome_arquivo_ler}', ele não existe")
    return dados_ler


In [4]:
#TESTE
teste_lercsv = ler_csv("transacoes.csv")
print(f"Total de linhas lidas: {len(teste_lercsv)}")
print(teste_lercsv[0])

Total de linhas lidas: 22
{'id': '1', 'data': '2026-01-05', 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': '3512.47', 'descricao': 'Salário', 'categoria': 'salario'}


In [5]:
# ----------------------------------------------------------------------
# Requisito: "Validação e limpeza dos dados"
# Antes de processar qualquer transação, valide cada linha.
# Descarte silenciosamente (sem parar o programa) as linhas que tiverem:
# - id vazio ou não numérico
# - cliente_id vazio
# - data em formato inválido (diferente de AAAA-MM-DD)
# - tipo diferente de credito ou debito
# - valor não numérico ou menor ou igual a zero
# ----------------------------------------------------------------------

def validar_id(id_texto):
    if not id_texto:
        return None
    try:
        return int(id_texto)
    except ValueError:
        return None


In [6]:
def validar_cliente_id(cliente_id_texto):
    if not cliente_id_texto:
        return None
    return cliente_id_texto

In [7]:
def validar_data(data_texto):
    try:
        return datetime.strptime(data_texto, "%Y-%m-%d")
    except ValueError:
        return None

In [8]:
def validar_tipo(tipo_texto):
    if tipo_texto not in ("credito", "debito"):
        return None
    return tipo_texto

In [9]:
def validar_valor(valor_texto):
    try:
        valor_convertido = float(valor_texto)
    except ValueError:
        return None
    if valor_convertido <= 0:
        return None
    return valor_convertido

In [10]:

def validar_transacao(transacao):
    id_convertido = validar_id(transacao.get("id"))
    if id_convertido is None:
        return None

    cliente_id = validar_cliente_id(transacao.get("cliente_id"))
    if cliente_id is None:
        return None

    data_convertida = validar_data(transacao.get("data"))
    if data_convertida is None:
        return None

    tipo = validar_tipo(transacao.get("tipo"))
    if tipo is None:
        return None

    valor_convertido = validar_valor(transacao.get("valor"))
    if valor_convertido is None:
        return None

    return {
        "id": id_convertido,
        "data": data_convertida,
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor_convertido,
        "descricao": transacao.get("descricao"),
        "categoria": transacao.get("categoria"),
    }

In [11]:
#Teste
print("transacao valida")
print(validar_transacao(teste_lercsv[0]))

print()
print("id nao numerico (ab) ")
print(validar_transacao(teste_lercsv[1]))

print()
print("cliente_id vazio")
print(validar_transacao(teste_lercsv[16]))

print()
print("id vazio")
print(validar_transacao(teste_lercsv[17]))

print()
print("data em formato invalido")
print(validar_transacao(teste_lercsv[18]))

print()
print("valor nao numérico (abc)")
print(validar_transacao(teste_lercsv[19]))

print()
print("tipo invalido (pix)")
print(validar_transacao(teste_lercsv[20]))

print()
print("valor negativo")
print(validar_transacao(teste_lercsv[21]))

transacao valida
{'id': 1, 'data': datetime.datetime(2026, 1, 5, 0, 0), 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': 3512.47, 'descricao': 'Salário', 'categoria': 'salario'}

id nao numerico (ab) 
None

cliente_id vazio
None

id vazio
None

data em formato invalido
None

valor nao numérico (abc)
None

tipo invalido (pix)
None

valor negativo
None


In [12]:
# ----------------------------------------------------------------------
# Continuação do requisito "Validação e limpeza dos dados":
# Ao final da leitura, exibe no terminal um resumo da limpeza.
# Usa continue para descartar silenciosamente uma linha inválida
# e seguir para a próxima, sem parar o programa.
# ----------------------------------------------------------------------
def processar_transacoes(transacoes_original):
    transacoes_validas = []
    total_invalidas = 0

    for transacao in transacoes_original:
        transacao_validada = validar_transacao(transacao)

        if transacao_validada is None:
            total_invalidas += 1
            continue

        transacoes_validas.append(transacao_validada)

    print("===== RESUMO DA LEITURA =====")
    print(f"Total de linhas lidas: {len(transacoes_original)}")
    print(f"Linhas válidas: {len(transacoes_validas)}")
    print(f"Linhas inválidas: {total_invalidas}")

    return transacoes_validas

In [13]:
#TESTE
transacoes_validas = processar_transacoes(teste_lercsv)

===== RESUMO DA LEITURA =====
Total de linhas lidas: 22
Linhas válidas: 15
Linhas inválidas: 7


In [14]:
def calcular_resumo_mensal(transacoes_validas):
    resumo_mensal = {}

    for transacao in transacoes_validas:
        mes = transacao["data"].strftime("%Y-%m")

        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "maior_valor": transacao["valor"],
                "menor_valor": transacao["valor"],
            }

        dados_mes = resumo_mensal[mes]
        dados_mes["quantidade"] += 1

        if transacao["tipo"] == "credito":
            dados_mes["total_credito"] += transacao["valor"]
        else:
            dados_mes["total_debito"] += transacao["valor"]

        if transacao["valor"] > dados_mes["maior_valor"]:
            dados_mes["maior_valor"] = transacao["valor"]
        if transacao["valor"] < dados_mes["menor_valor"]:
            dados_mes["menor_valor"] = transacao["valor"]

    for mes, dados_mes in resumo_mensal.items():
        dados_mes["saldo"] = dados_mes["total_credito"] - dados_mes["total_debito"]
        dados_mes["media"] = (dados_mes["total_credito"] + dados_mes["total_debito"]) / dados_mes["quantidade"]

    return resumo_mensal

In [15]:
def identificar_transacoes_suspeitas(transacoes_validas):
    transacoes_suspeitas = []

    for transacao in transacoes_validas:
        if transacao["valor"] > LIMITE_SUSPEITO:
            transacoes_suspeitas.append({
                "id": transacao["id"],
                "cliente_id": transacao["cliente_id"],
                "data": transacao["data"].strftime("%Y-%m-%d"),
                "valor": transacao["valor"],
            })

    return transacoes_suspeitas


In [16]:
def calcular_periodo(transacoes_validas):
    data_mais_antiga = None
    data_mais_recente = None

    for transacao in transacoes_validas:
        data_transacao = transacao["data"]
        if data_mais_antiga is None or data_transacao < data_mais_antiga:
            data_mais_antiga = data_transacao
        if data_mais_recente is None or data_transacao > data_mais_recente:
            data_mais_recente = data_transacao

    dias_periodo = (data_mais_recente - data_mais_antiga).days

    return {
        "data_mais_antiga": data_mais_antiga,
        "data_mais_recente": data_mais_recente,
        "dias_periodo": dias_periodo,
    }

In [17]:
def gerar_relatorio(transacoes_validas):
    relatorio = {
        "resumo_mensal": calcular_resumo_mensal(transacoes_validas),
        "transacoes_suspeitas": identificar_transacoes_suspeitas(transacoes_validas),
        "periodo": calcular_periodo(transacoes_validas),
    }
    return relatorio

In [18]:
def formatar_moeda(valor):
    texto = f"R$ {valor:,.2f}"
    texto = texto.replace(",", "X").replace(".", ",").replace("X", ".")
    return texto

In [19]:
#TESTE
total_validas = len(transacoes_validas)
total_invalidas = len(teste_lercsv) - len(transacoes_validas)

In [20]:
# ----------------------------------------------------------------------
# Requisito: "Exibição formatada no terminal"
# Separadores visuais entre seções, valores monetários com 2 casas
# decimais e prefixo R$, período analisado, total de válidas/inválidas.
# ----------------------------------------------------------------------
def exibir_relatorio(relatorio, total_validas, total_invalidas):
    periodo = relatorio["periodo"]
    data_antiga = periodo["data_mais_antiga"].strftime("%Y-%m-%d")
    data_recente = periodo["data_mais_recente"].strftime("%Y-%m-%d")

    print(f"Período analisado: {data_antiga} até {data_recente}")
    print(f"Total de transações válidas: {total_validas}")
    print(f"Total de transações inválidas: {total_invalidas}")
    print()

    for mes, dados_mes in relatorio["resumo_mensal"].items():
        print("===== RELATÓRIO MENSAL =====")
        print()
        print(f"Mês: {mes}")
        print(f"  Transações: {dados_mes['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(dados_mes['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(dados_mes['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(dados_mes['saldo'])}")
        print(f"  Média:         {formatar_moeda(dados_mes['media'])}")
        print(f"  Maior valor:   {formatar_moeda(dados_mes['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(dados_mes['menor_valor'])}")
        print()

    print("===== TRANSAÇÕES SUSPEITAS =====")
    transacoes_suspeitas = relatorio["transacoes_suspeitas"]
    if len(transacoes_suspeitas) == 0:
        print("Nenhuma transação suspeita encontrada.")
    else:
        for suspeita in transacoes_suspeitas:
            print(f"ID: {suspeita['id']} | Cliente: {suspeita['cliente_id']} | Data: {suspeita['data']} | Valor: {formatar_moeda(suspeita['valor'])}")

In [21]:
#TESTE
relatorio = gerar_relatorio(transacoes_validas)
exibir_relatorio(relatorio, total_validas, total_invalidas)

Período analisado: 2026-01-05 até 2026-04-04
Total de transações válidas: 15
Total de transações inválidas: 7

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações: 3
  Total crédito: R$ 3.512,47
  Total débito:  R$ 111,70
  Saldo:         R$ 3.400,77
  Média:         R$ 1.208,06
  Maior valor:   R$ 3.512,47
  Menor valor:   R$ 42,80

===== RELATÓRIO MENSAL =====

Mês: 2026-02
  Transações: 5
  Total crédito: R$ 18.020,50
  Total débito:  R$ 665,34
  Saldo:         R$ 17.355,16
  Média:         R$ 3.737,17
  Maior valor:   R$ 15.230,90
  Menor valor:   R$ 54,60

===== RELATÓRIO MENSAL =====

Mês: 2026-03
  Transações: 6
  Total crédito: R$ 6.302,07
  Total débito:  R$ 12.374,72
  Saldo:         R$ -6.072,65
  Média:         R$ 3.112,80
  Maior valor:   R$ 12.045,80
  Menor valor:   R$ 33,90

===== RELATÓRIO MENSAL =====

Mês: 2026-04
  Transações: 1
  Total crédito: R$ 0,00
  Total débito:  R$ 94,30
  Saldo:         R$ -94,30
  Média:         R$ 94,30
  Maior valor:   R$ 94,30
  Men

In [22]:
# ----------------------------------------------------------------------
# Requisito: "Exportação do relatório em JSON"
# Salva o relatório no arquivo relatorio.json com a estrutura definida
# no enunciado (gerado_em, totais, resumo_mensal, transacoes_suspeitas).
# ----------------------------------------------------------------------
def salvar_json(relatorio, total_validas, total_invalidas, nome_arquivo="relatorio.json"):
    resumo_mensal_formatado = {}
    for mes, dados_mes in relatorio["resumo_mensal"].items():
        resumo_mensal_formatado[mes] = {
            "quantidade": dados_mes["quantidade"],
            "total_credito": round(dados_mes["total_credito"], 2),
            "total_debito": round(dados_mes["total_debito"], 2),
            "saldo": round(dados_mes["saldo"], 2),
            "media": round(dados_mes["media"], 2),
            "maior_valor": round(dados_mes["maior_valor"], 2),
            "menor_valor": round(dados_mes["menor_valor"], 2),
        }

    dados_json = {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "total_transacoes_validas": total_validas,
        "total_transacoes_invalidas": total_invalidas,
        "resumo_mensal": resumo_mensal_formatado,
        "transacoes_suspeitas": relatorio["transacoes_suspeitas"],
    }

    with open(nome_arquivo, mode="w", encoding="utf-8") as arquivo:
        json.dump(dados_json, arquivo, ensure_ascii=False, indent=2)

    print(f"Arquivo '{nome_arquivo}' salvo com sucesso.")
    return dados_json

In [23]:
#TESTE
dados_salvos = salvar_json(relatorio, total_validas, total_invalidas)
print(json.dumps(dados_salvos, ensure_ascii=False, indent=2))

Arquivo 'relatorio.json' salvo com sucesso.
{
  "gerado_em": "2026-07-23",
  "total_transacoes_validas": 15,
  "total_transacoes_invalidas": 7,
  "resumo_mensal": {
    "2026-01": {
      "quantidade": 3,
      "total_credito": 3512.47,
      "total_debito": 111.7,
      "saldo": 3400.77,
      "media": 1208.06,
      "maior_valor": 3512.47,
      "menor_valor": 42.8
    },
    "2026-02": {
      "quantidade": 5,
      "total_credito": 18020.5,
      "total_debito": 665.34,
      "saldo": 17355.16,
      "media": 3737.17,
      "maior_valor": 15230.9,
      "menor_valor": 54.6
    },
    "2026-03": {
      "quantidade": 6,
      "total_credito": 6302.07,
      "total_debito": 12374.72,
      "saldo": -6072.65,
      "media": 3112.8,
      "maior_valor": 12045.8,
      "menor_valor": 33.9
    },
    "2026-04": {
      "quantidade": 1,
      "total_credito": 0.0,
      "total_debito": 94.3,
      "saldo": -94.3,
      "media": 94.3,
      "maior_valor": 94.3,
      "menor_valor": 94.3
  

In [24]:
# ----------------------------------------------------------------------
# ==============Célula de Execução Principal==============
# Executa o pipeline completo do projeto:
# leitura -> validação -> relatório -> exportação em JSON -> exibição no terminal
# ========================================================
# ----------------------------------------------------------------------

# 1. Leitura do CSV
teste_lercsv = ler_csv("transacoes.csv")

# 2. Validação e limpeza (usa validar_transacao internamente)
transacoes_validas = processar_transacoes(teste_lercsv)

# 3. Geração do relatório (agrupamento mensal, suspeitas, período)
relatorio = gerar_relatorio(transacoes_validas)

# 4. Totais para exportação e exibição
total_validas = len(transacoes_validas)
total_invalidas = len(teste_lercsv) - len(transacoes_validas)

# 5. Exportação em JSON
salvar_json(relatorio, total_validas, total_invalidas)

# 6. Exibição formatada no terminal
print()
exibir_relatorio(relatorio, total_validas, total_invalidas)

===== RESUMO DA LEITURA =====
Total de linhas lidas: 22
Linhas válidas: 15
Linhas inválidas: 7
Arquivo 'relatorio.json' salvo com sucesso.

Período analisado: 2026-01-05 até 2026-04-04
Total de transações válidas: 15
Total de transações inválidas: 7

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações: 3
  Total crédito: R$ 3.512,47
  Total débito:  R$ 111,70
  Saldo:         R$ 3.400,77
  Média:         R$ 1.208,06
  Maior valor:   R$ 3.512,47
  Menor valor:   R$ 42,80

===== RELATÓRIO MENSAL =====

Mês: 2026-02
  Transações: 5
  Total crédito: R$ 18.020,50
  Total débito:  R$ 665,34
  Saldo:         R$ 17.355,16
  Média:         R$ 3.737,17
  Maior valor:   R$ 15.230,90
  Menor valor:   R$ 54,60

===== RELATÓRIO MENSAL =====

Mês: 2026-03
  Transações: 6
  Total crédito: R$ 6.302,07
  Total débito:  R$ 12.374,72
  Saldo:         R$ -6.072,65
  Média:         R$ 3.112,80
  Maior valor:   R$ 12.045,80
  Menor valor:   R$ 33,90

===== RELATÓRIO MENSAL =====

Mês: 2026-04
  Transaçõe